# SAM3 SageMaker real-time endpoint

This notebook packages a fine-tuned (or pretrained) SAM3 checkpoint and
deploys it as a SageMaker real-time endpoint, then invokes it with a
text-prompt request and a click-prompt request.

**Inputs** (local):

- `CHECKPOINT_PATH` → local SAM3 checkpoint (e.g. the merged file that
  the training notebook produced). Set to `None` to deploy `facebook/sam3`
  and let the handler pull weights from HuggingFace on container boot.
- `BPE_PATH`        → `sam3/assets/bpe_simple_vocab_16e6.txt.gz`
- `INFERENCE_DIR`   → `sagemaker/deploy/` (contains `inference.py` +
  `requirements.txt`; both get packaged under `code/` in the tarball).

**Outputs**:

- `s3://<bucket>/<prefix>/model.tar.gz` — the model bundle SageMaker
  imports on container boot.
- A running endpoint at `<ENDPOINT_NAME>` (billed hourly until deleted).

## Prerequisites

```bash
pip install 'sagemaker>=2.230,<3' boto3 pycocotools pillow
aws configure   # or assume an IAM role with sagemaker:*, iam:PassRole, s3 access
```

The IAM role passed to the endpoint (`ROLE`) needs `AmazonS3ReadOnlyAccess`
on the bucket + the standard SageMaker execution policy.

**Cost warning**: an `ml.g5.xlarge` endpoint is ~$1.40/hr on-demand.
Section 5 deletes the endpoint — run it when done.

In [ ]:
# Pin sagemaker<3 — the v3 SDK is a major rewrite that moves PyTorchModel /
# Session / get_execution_role out of their canonical paths. This notebook
# targets the long-stable v2 API (same as launch_training.ipynb).
%pip install --quiet 'sagemaker>=2.230,<3' boto3 pycocotools pillow
# IMPORTANT: if the kernel already imported a v3 build, restart it before
# running the next cell.

In [ ]:
# ---------- Configuration: edit these ----------
import os
import sagemaker
import boto3

_v = getattr(sagemaker, "__version__", "unknown")
if _v.startswith("3.") or _v.startswith("4."):
    raise RuntimeError(
        f"Detected sagemaker {_v}, but this notebook requires v2.x. "
        f"Run the previous cell, then RESTART THE KERNEL, then retry."
    )
print("sagemaker version:", _v)

from sagemaker import get_execution_role

sess   = sagemaker.Session()
role   = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_session.region_name
print("sagemaker_default_bucket:", bucket)
print("sagemaker_region:", region)

# ---- Repo + local artifact paths ----
LOCAL_REPO_ROOT = "/home/ec2-user/SageMaker/efs/Projects/sam3"
# Set to a local .pt file to bundle a fine-tuned checkpoint, or leave as
# None to deploy facebook/sam3 unmodified (handler auto-pulls from HF).
CHECKPOINT_PATH = os.path.join(
    LOCAL_REPO_ROOT, "runs/aws_sam_finetune/checkpoints/checkpoint_merged.pt"
)
# CHECKPOINT_PATH = None   # ← uncomment to deploy pretrained SAM3

BPE_PATH       = os.path.join(LOCAL_REPO_ROOT, "sam3/assets/bpe_simple_vocab_16e6.txt.gz")
INFERENCE_DIR  = os.path.join(LOCAL_REPO_ROOT, "sagemaker/deploy")

# ---- S3 layout ----
S3_PREFIX  = "projects/sam3/deploy/aws_sam_v1"
S3_TARBALL = f"s3://{bucket}/{S3_PREFIX}/model.tar.gz"

# ---- Endpoint settings ----
INSTANCE_TYPE      = "ml.g5.xlarge"    # 1 × A10G 24GB — comfortable for 1008×1008 inference
INSTANCE_COUNT     = 1
ENDPOINT_NAME      = "sam3-aws-sam"    # must be unique per region
FRAMEWORK_VERSION  = "2.3.0"
PY_VERSION         = "py311"

# Local scratch for the assembled tarball
LOCAL_TARBALL = os.path.join(LOCAL_REPO_ROOT, "tmp", "s3_sagemaker_deploy", "model.tar.gz")

print("checkpoint:  ", CHECKPOINT_PATH or "(pretrained facebook/sam3 from HF)")
print("bpe:         ", BPE_PATH)
print("inference:   ", INFERENCE_DIR)
print("s3 tarball:  ", S3_TARBALL)
print("endpoint:    ", ENDPOINT_NAME, "on", INSTANCE_TYPE)

## 1. Build the model tarball

SageMaker's PyTorch inference container expects this layout inside
`model.tar.gz`:

```
checkpoint.pt                       # optional — handler falls back to HF
bpe_simple_vocab_16e6.txt.gz
code/
  inference.py                      # model_fn / input_fn / predict_fn / output_fn
  requirements.txt                  # extra pip deps installed on container boot
```

The staging + tarball assembly happens in `tmp/s3_sagemaker_deploy/`
(under `.gitignore`).

In [ ]:
import shutil
import tarfile
from pathlib import Path

STAGING_DIR = os.path.join(LOCAL_REPO_ROOT, "tmp", "s3_sagemaker_deploy", "staging")
if os.path.exists(STAGING_DIR):
    shutil.rmtree(STAGING_DIR)
os.makedirs(STAGING_DIR)

# Top-level: checkpoint + BPE vocab
if CHECKPOINT_PATH is not None:
    assert os.path.exists(CHECKPOINT_PATH), f"missing checkpoint: {CHECKPOINT_PATH}"
    shutil.copy(CHECKPOINT_PATH, os.path.join(STAGING_DIR, "checkpoint.pt"))
    print(f"  + checkpoint.pt  ({os.path.getsize(CHECKPOINT_PATH)/1e9:.2f} GB)")
else:
    print("  (no checkpoint bundled — handler will pull facebook/sam3 from HF)")
shutil.copy(BPE_PATH, os.path.join(STAGING_DIR, os.path.basename(BPE_PATH)))
print(f"  + {os.path.basename(BPE_PATH)}")

# code/ subdir — SageMaker auto-installs requirements.txt from here and
# imports inference.py.
code_dst = os.path.join(STAGING_DIR, "code")
os.makedirs(code_dst)
for fname in ("inference.py", "requirements.txt"):
    src = os.path.join(INFERENCE_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(code_dst, fname))
        print(f"  + code/{fname}")
    else:
        print(f"  ! code/{fname} missing at {src}")

# Tar it up.
os.makedirs(os.path.dirname(LOCAL_TARBALL), exist_ok=True)
if os.path.exists(LOCAL_TARBALL):
    os.remove(LOCAL_TARBALL)
with tarfile.open(LOCAL_TARBALL, "w:gz") as tf:
    for p in Path(STAGING_DIR).rglob("*"):
        tf.add(p, arcname=p.relative_to(STAGING_DIR))

print(f"\nlocal tarball: {LOCAL_TARBALL}")
print(f"size:          {os.path.getsize(LOCAL_TARBALL)/1e9:.2f} GB")

## 2. Upload the tarball to S3

In [ ]:
from sagemaker.s3 import S3Uploader

boto_session = boto3.Session(region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session)

model_data = S3Uploader.upload(
    local_path=LOCAL_TARBALL,
    desired_s3_uri=f"s3://{bucket}/{S3_PREFIX}",
    sagemaker_session=sagemaker_session,
)
print("uploaded:", model_data)

## 3. Deploy the endpoint

SageMaker downloads the tarball, extracts it to `/opt/ml/model/`, pip-installs
`code/requirements.txt`, imports `code/inference.py`, and calls `model_fn`
once per worker. First-boot pip install + weight load takes ~10 minutes
for SAM3; subsequent restarts on the same instance are much faster.

If an endpoint with `ENDPOINT_NAME` already exists, SageMaker replaces it
in place (a rolling blue/green update).

In [ ]:
from sagemaker.pytorch import PyTorchModel

model = PyTorchModel(
    model_data=model_data,
    role=role,
    framework_version=FRAMEWORK_VERSION,
    py_version=PY_VERSION,
    entry_point="inference.py",
    sagemaker_session=sagemaker_session,
    env={
        # Pre-cache HF home so first request doesn't time out if the handler
        # decides to fetch facebook/sam3 (only happens when CHECKPOINT_PATH
        # is None). Set HF_TOKEN here only if the endpoint role can't read
        # from HF unauthenticated.
        "TRANSFORMERS_CACHE": "/opt/ml/model/.hf_cache",
        "HF_HOME":            "/opt/ml/model/.hf_cache",
    },
)

predictor = model.deploy(
    initial_instance_count=INSTANCE_COUNT,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", predictor.endpoint_name)

## 4. Smoke-test the endpoint

Two request modes:

- **text**    — give the model a noun phrase, it returns 0..N masks.
- **click**   — SAM-1-style interactive: points + labels, up to 3 masks back.

Masks come back as COCO-RLE. `pycocotools.mask.decode` reconstructs the
binary mask locally.

In [ ]:
# ---- Text-prompt request ----
import base64
import json

IMAGE_PATH = os.path.join(LOCAL_REPO_ROOT, "data/AWS_SAM/companypremises2025101600217.png")
TEXT_PROMPT = "grass"

with open(IMAGE_PATH, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode("ascii")

body = {
    "mode": "text",
    "image_b64": img_b64,
    "text": TEXT_PROMPT,
    "confidence": 0.5,
}

smr = boto_session.client("sagemaker-runtime")
resp = smr.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(body),
)
out = json.loads(resp["Body"].read())
print(f"text='{TEXT_PROMPT}'  n_masks={len(out['masks_rle'])}  scores={out['scores']}")
print(f"image_size={out['image_size']}")

In [ ]:
# ---- Click-prompt request ----
CLICK_XY = [520, 375]     # (x, y) pixel, positive click

body = {
    "mode": "click",
    "image_b64": img_b64,
    "points": [CLICK_XY],
    "labels": [1],                # 1 = positive, 0 = negative
    "multimask_output": True,     # returns 3 candidate masks
}
resp = smr.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(body),
)
out = json.loads(resp["Body"].read())
print(f"click={CLICK_XY}  n_masks={len(out['masks_rle'])}  scores={out['scores']}")

In [ ]:
# ---- Decode + visualize the returned masks ----
import numpy as np
from PIL import Image
from pycocotools import mask as mask_utils
import matplotlib.pyplot as plt

image = np.array(Image.open(IMAGE_PATH).convert("RGB"))
masks = [mask_utils.decode(rle).astype(bool) for rle in out["masks_rle"]]

fig, axes = plt.subplots(1, max(1, len(masks)), figsize=(4 * max(1, len(masks)), 4))
if len(masks) <= 1:
    axes = [axes]
for ax, m, s in zip(axes, masks, out["scores"]):
    overlay = image.copy()
    overlay[m] = (0.5 * overlay[m] + 0.5 * np.array([255, 60, 60])).astype(np.uint8)
    ax.imshow(overlay)
    ax.scatter([CLICK_XY[0]], [CLICK_XY[1]], c="lime", s=60, marker="*", edgecolors="black")
    ax.set_title(f"score={s:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Cleanup

**Endpoints are billed hourly until deleted.** Run this cell when you're
done — leaving `ml.g5.xlarge` running is ~$34/day.

In [ ]:
# Delete the endpoint + its config + the model resource.
sm = boto_session.client("sagemaker")
for op in (
    ("delete_endpoint",         {"EndpointName":       ENDPOINT_NAME}),
    ("delete_endpoint_config",  {"EndpointConfigName": ENDPOINT_NAME}),
    ("delete_model",            {"ModelName":          predictor.model_name if 'predictor' in dir() else ENDPOINT_NAME}),
):
    try:
        getattr(sm, op[0])(**op[1])
        print(f"✓ {op[0]}", op[1])
    except sm.exceptions.ClientError as e:
        print(f"  skip {op[0]}: {e}")

# S3 tarball stays put — uncomment to remove:
# import boto3
# boto3.client("s3").delete_object(Bucket=bucket, Key=f"{S3_PREFIX}/model.tar.gz")

## Notes / gotchas

- **First boot is slow** because `sam3` gets pip-installed from GitHub
  inside the container (see `requirements.txt`). If you deploy often,
  build a custom Docker image with `sam3` baked in and pass `image_uri=...`
  to `PyTorchModel(...)` instead of `framework_version`.
- **`entry_point="inference.py"`** — SageMaker resolves this against
  the `code/` subdir it extracts from the tarball, NOT against a local
  filesystem path. You don't need to pass `source_dir` because the
  package layout already puts everything under `code/`.
- **HuggingFace download**: if `CHECKPOINT_PATH is None`, `model_fn`
  calls `build_sam3_image_model(load_from_HF=True, ...)`. The endpoint
  role needs internet access + `TRANSFORMERS_CACHE` set (done above),
  and optionally `HF_TOKEN` for gated repos.
- **Cold vs warm invocations**: the first request after deploy runs
  `torch.compile`-free but still burns ~2–4 s in autograd/CUDNN setup.
  Warm p50 for a 1008×1008 text prompt is ~200–300 ms on `ml.g5.xlarge`.
- **Autoscaling**: for production, wrap this endpoint with an
  application-autoscaling target policy (SageMaker docs: "Add automatic
  scaling to a model deployed to a real-time endpoint").
- **Multi-model endpoints** are NOT supported here — the SAM3 model
  is stateful (`model_fn` returns a live processor) and multi-model
  endpoints re-instantiate models per invocation.